# Gemini API를 활용한 등산 후기 분석

**목표:**

- 전체 등산 후기 중 실제 등산 후기만 자동 필터링

- 등산 후기의 매력 요소(뷰·힐링·난이도 등)를 항목별 점수화(1~10)

- 후기에서 핵심 키워드·감정·신뢰도 자동 추출

- 다수 후기를 배치 분석하여 평균 지표 및 분포 도출

- Gemini API를 활용해 산·코스별 정량 비교와 인사이트 확보

- 텍스트 길이가 짧은 유튜브 댓글과 지도 리뷰는 산별로 20개씩 묶어 분석

In [ ]:
# 필요한 라이브러리 import
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Any
import enum
from datetime import datetime
from pprint import pprint
import time
import json
import math

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL')

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {gemini_model}")

# 클라이언트 초기화
client = genai.Client(api_key=api_key)

# 프롬프트

In [ ]:
filter_system_instruction = """
당신은 등산 및 트레킹 후기 텍스트를 **필터링**하는 '데이터 분석 전문가'입니다.
주어진 한국어 블로그 후기를 읽고,
1) 이 텍스트가 **실질적인 등산 후기인지 여부**만 판단하여,
2) 아래 JSON 형식으로만 출력하세요.

[등산후기 판별 기준]
- `is_hiking_review`는 다음 기준으로 판단합니다.
  - true  : 등산/코스/산행 경험 묘사가 본문 내용의 **절반 이상**을 차지한다고 볼 수 있을 때
  - false : 음식점·카페·숙소 등 가게 후기이거나, 등산 언급이 곁다리 수준일 때

[중요]
- 출력에는 **반드시 다음 두 개 필드만 포함**해야 합니다.
  - `review_id`
  - `is_hiking_review`

[출력 형식]
- **반드시 순수 JSON 문자열만** 출력합니다. (마크다운 코드블록 사용 금지)
- 모든 Key와 문자열 Value는 큰따옴표("")로 감싸야 합니다.

[출력 예시]
{
  "review_id": "가리산_naver_blog_1",
  "is_hiking_review": true
}
"""

analysis_system_instruction = """
당신은 등산 및 트레킹 후기 텍스트를 정밀 분석하는 '데이터 분석 전문가'입니다.
주어진 한국어 블로그 후기를 분석하여, 산과 코스의 매력도를 정해진 JSON 포맷으로 추출하세요.

[분석 가이드라인]

1. 리뷰 메타데이터 (review_id)
   - 입력 텍스트와 함께 ID가 주어지면 해당 ID를 사용. 없으면 null.

   1-1. 전체 핵심 키워드 (overall_keywords)
   - 해당 산/코스의 전반적인 특징을 가장 잘 요약하는 명사/명사구를 최대 5개 추출
   - 키워드 예시: ["뷰맛집", "힐링코스", "초보추천", "단풍명소", "암릉구간"]
   - 항목별 키워드와 중복되어도 무관하며, 후기 전체를 대표하는 핵심 키워드를 선정
   
2. 매력 포인트 점수 및 키워드 (1~10점 척도 + 항목별 키워드)
   - 각 항목에 대해 **1(매우 약함/나쁨) ~ 10(매우 강함/최고)** 사이의 정수로 평가하세요.
   
      --- **점수 해석 상세 가이드라인 (Score Interpretation)** ---
      - **1~3점**: 매우 약하거나 부정적 (예: "너무 별로였어요.", "완전 실망.", "절대 오지 마세요.")
      - **4~6점**: 평이함, 보통, 기대 이하 (예: "그냥 그럭저럭", "딱히 좋지도 나쁘지도 않음.", "살짝 아쉽네요.")
      - **7~8점**: 좋음, 만족스러움 (예: "꽤 좋았어요!", "추천합니다.", "만족스러운 수준.")
      - **9~10점**: 매우 강함, 최고, 감탄 (예: "인생 최고!", "강력 추천!", "힐링 그 자체.", "뷰 미쳤어요.")
      - 해당 내용이 전혀 언급되지 않았거나, 판단 근거가 부족하면 점수는 `null`, 키워드는 빈 배열 `[]`로 설정하세요.
      - **각 항목별로 해당하는 키워드를 최대 3개까지 추출하세요.**
   
   항목별 분석:
   
   **view_score & view_keywords**:
   - 점수: 뷰·경관 (풍경, 전망, 일출·일몰, 운해, 조망)
   - 키워드 예시: ["정상뷰", "일출명소", "운해"]
   
   **healing_score & healing_keywords**:
   - 점수: 힐링·분위기 (피톤치드, 숲 냄새, 조용함, 숲길, 자연의 소리)
   - 키워드 예시: ["피톤치드", "힐링코스", "숲길"]
   
   **sns_photo_score & sns_keywords**:
   - 점수: 사진·SNS (인생샷, 포토존, 사진 잘 나오는 곳)
   - 키워드 예시: ["인생샷", "포토존", "사진명소"]
   
   **trail_condition_score & trail_keywords**:
   - 점수: 등산로 상태 (정비 상태, 이정표, 매트, 시설 청결도)
   - 키워드 예시: ["등산로정비", "안전시설", "이정표"]
   
   **fun_achievement_score & fun_keywords**:
   - 점수: 재미·성취감 (암릉, 스릴, 난이도 극복, 운동 강도)
   - 키워드 예시: ["암릉구간", "등반난이도", "성취감"]
   
   **seasonal_attraction_score & seasonal_keywords**:
   - 점수: 계절적 매력 (봄꽃, 단풍, 설경, 상고대 등 특정 계절 요소)
   - 키워드 예시: ["단풍명소", "설경", "벚꽃"]
   
   **accessibility_score & accessibility_keywords**:
   - 점수: 접근 편의성 (대중교통, 버스, 셔틀버스, 주차장 편의성, 지하철역, 입구까지 거리)
   - 키워드 예시: ["가까움", "대중교통편리", "주차혼잡", "접근성", "정류장"]
   - **중요**: 교통/접근성에 대한 구체적 언급이 없으면 반드시 `null`로 설정
   
3. 감정 (sentiment_label)
   - **positive**: 긍정 (추천, 재방문 의사, 만족, 행복)
   - **neutral**: 중립 (단순 정보 전달, 감정 표현 절제)
   - **negative**: 부정 (비추천, 실망, 불만, 위험 경고)

4. 신뢰도 (reliability)
   - **high**: 내용이 구체적이고 길며, 직접 경험한 디테일이 풍부함
   - **medium**: 정보는 있으나 평이하거나 짧음
   - **low**: 내용이 빈약하거나 광고성/복사 붙여넣기 의심

[주의사항]
- **한국어 뉘앙스 파악**: "조금 아쉽다(부정 힌트)", "힘들지만 갈만하다(성취감 긍정)" 등 완곡한 표현과 이모티콘, 문맥을 깊이 있게 고려하세요.
- **키워드 추출**: 각 항목의 키워드는 해당 매력 포인트를 가장 잘 설명하는 명사/명사구로 구성하세요.
- **출력 형식**: 순수 JSON 문자열만 출력하세요.
- **JSON 문법 준수**: 모든 키(Key)와 문자열 값(Value)은 반드시 큰따옴표("")로 감싸야 합니다.

[출력 예시]
{
  "review_id": "가리산_naver_blog_1",
  "overall_keywords": ["뷰맛집", "힐링코스", "초보추천", "단풍명소", "가을산행"],  
  "view_score": 9,
  "view_keywords": ["정상뷰", "운해", "일출명소"],
  "healing_score": 8,
  "healing_keywords": ["숲길", "피톤치드"],
  "sns_photo_score": 8,
  "sns_keywords": ["인생샷", "포토존"],
  "trail_condition_score": 7,
  "trail_keywords": ["등산로정비", "이정표"],
  "fun_achievement_score": null,
  "fun_keywords": [],
  "seasonal_attraction_score": 10,
  "seasonal_keywords": ["단풍명소", "가을산행", "단풍터널"],
  "accessibility_score": 2,
  "accessibility_keywords": ["주차장없음", "도로혼잡", "주차힘듦"],  
  "sentiment_label": "positive",
  "reliability": "high"
}

"""

filter_system_instruction_youtube = """
당신은 등산 및 트레킹 정보 텍스트를 **필터링**하는 '데이터 분석 전문가'입니다.
주어진 한국어 댓글을 읽고,
1) 이 텍스트가 **실제 등산/트레킹 경험이나 산 관련 정보 내용 인지 여부**만 판단하여,
2) 아래 JSON 형식으로만 출력하세요.

[판별 기준]
True : 댓글 내용 중 산행/등산 경험, 코스, 난이도, 위험 구간, 풍경, 힐링, 재미, 성취감, SNS 사진 경험 등 **직접 경험 기반 산 관련 정보**가 포함된 경우
False : 산과 무관한 내용, 단순 감사/칭찬/인사, 감탄사, 출연자 평가, 영상 편집 평가, 개인 이야기 등이 **내용의 50% 이상 ** 차지하는 경우

[True 포함 내용 구체화]
- 산/등산 경험, 코스, 난이도, 위험 구간
- 주변 맛집, 카페, 숙소, 주차장, 관광지
- 풍경, 계절, 힐링, 재미, 성취감, SNS 사진 경험
- 직접 경험 기반 리뷰


[False 포함 내용 구체화]
- "좋아요", "잘봤다", "감사합니다", "도움 됐다", "재밌다", "멋지다" 등 단순 칭찬/감사 표현
- 유튜버/출연자 관련: 외모, 성격, 말투, 몸매, 인기, 구독 요청, 타 유튜버/타 영상 비교
- 영상 제작 관련: 편집, 촬영, 음악, 기획, 자막, 촬영 기술 평가
- 개인적/무관한 내용: 장소와 무관한 개인 이야기, 신변 이야기, 직접 경험 없는 추측성 발언
- 홍보/광고/스팸: 장비·브랜드 홍보, 광고, 반복 문구, AI 의심 일반론
- 무의미한 댓글: 감탄사 중심, 웃음표현, 이모지 중심, 정보 없는 단문
- 정치/사회/종교 등 논쟁 가능 주제

[중요]
- 출력에는 **반드시 다음 두 개 필드만 포함**해야 합니다.
  - `review_id`
  - `is_youtube_review`

  
  [출력 형식]
- **반드시 순수 JSON 문자열만** 출력합니다. (마크다운 코드블록 사용 금지)
- 모든 Key와 문자열 Value는 큰따옴표("")로 감싸야 합니다.

[출력 예시]
{
  "review_id": "가리산_youtube_1",
  "is_hiking_review": true
}

"""

analysis_system_instruction_youtube = """
당신은 등산 및 트레킹 리뷰 텍스트를 분석하는 전문가입니다.
주어진 입력은 **동일 산에 대한 20여개의 짧은 리뷰를 하나로 합친 텍스트**입니다.
각 리뷰는 서로 다른 사람이 작성했으며, 의견이 상충될 수 있습니다.
주어진 한국어 후기를 분석하여, 산과 코스의 매력도를 정해진 JSON 포맷으로 추출하세요.

[분석 가이드라인]

1. 리뷰 메타데이터 (review_id)
   - 입력 텍스트와 함께 ID가 주어지면 해당 ID를 사용. 없으면 null.

   1-1. 전체 핵심 키워드 (overall_keywords)
   - 해당 산/코스의 전반적인 특징을 가장 잘 요약하는 명사/명사구를 최대 5개 추출
   - 키워드 예시: ["뷰맛집", "힐링코스", "초보추천", "단풍명소", "암릉구간"]
   - 항목별 키워드와 중복되어도 무관하며, 후기 전체를 대표하는 핵심 키워드를 선정
   
2. 매력 포인트 점수 및 키워드 (1~10점 척도 + 항목별 키워드)
   - 각 항목에 대해 **1(매우 약함/나쁨) ~ 10(매우 강함/최고)** 사이의 정수로 평가하세요.
   
      --- **점수 해석 상세 가이드라인 (Score Interpretation)** ---
      - **1~3점**: 매우 약하거나 부정적 (예: "별로였어요.", "실망.", "재미없어요")
      - **4~6점**: 평이함, 보통, 기대 이하 (예: "그냥 그럭저럭", "딱히 좋지도 나쁘지도 않음.", "살짝 아쉽네요.")
      - **7~8점**: 좋음, 만족스러움 (예: "꽤 좋았어요!", "추천합니다.", "만족스러운 수준.")
      - **9~10점**: 매우 강함, 최고, 감탄 (예: "인생 산", "강력 추천!", "힐링 그 자체.", "뷰 미쳤어요.")
      - 해당 내용이 전혀 언급되지 않았거나, 판단 근거가 부족하면 점수는 `null`, 키워드는 빈 배열 `[]`로 설정하세요.
      - **각 항목별로 해당하는 키워드를 최대 3개까지 추출하세요.**
   
   항목별 분석:

   **view_score & view_keywords**:
   - 점수: 뷰·경관 (풍경, 전망, 일출·일몰, 운해, 조망)
   - 키워드 예시: ["정상뷰", "일출명소", "운해"]
   
   **healing_score & healing_keywords**:
   - 점수: 힐링·분위기 (피톤치드, 숲 냄새, 조용함, 숲길, 자연의 소리)
   - 키워드 예시: ["피톤치드", "힐링코스", "숲길"]
   
   **sns_photo_score & sns_keywords**:
   - 점수: 사진·SNS (인생샷, 포토존, 사진 잘 나오는 곳)
   - 키워드 예시: ["인생샷", "포토존", "사진명소"]
   
   **trail_condition_score & trail_keywords**:
   - 점수: 등산로 상태 (정비 상태, 이정표, 시설 청결도)
   - 키워드 예시: ["정비", "안전시설", "이정표", "파손", "노후"]
   
   **fun_achievement_score & fun_keywords**:
   - 점수: 재미·성취감 (암릉, 스릴, 난이도 극복, 운동 강도)
   - 키워드 예시: ["암릉구간", "등반난이도", "성취감"]
   
   **seasonal_attraction_score & seasonal_keywords**:
   - 점수: 계절적 매력 (봄꽃, 단풍, 설경, 상고대 등 특정 계절 요소)
   - 키워드 예시: ["단풍명소", "설경", "벚꽃"]
   
   **accessibility_score & accessibility_keywords**:
   - 점수: 접근 편의성 (대중교통, 버스, 셔틀버스, 주차장 편의성, 지하철역, 입구까지 거리)
   - 키워드 예시: ["가까움", "대중교통편리", "주차혼잡", "접근성", "정류장"]
   
   [접근성 평가 기준]
   접근성은 "등산로 입구까지 가는 편의성"만 평가합니다.
   - 긍정 요소 (점수 UP): 주차장이 등산로 입구와 가까움, 주차 공간이 넉넉함, 대중교통 접근 가능, 셔틀버스 운행
   - 부정 요소 (점수 DOWN): 주차장이 등산로에서 멀리 떨어짐, 주차 공간 협소, 대중교통 없음, 주차장 없음
   - 중요: 등산 소요시간과 혼동하지 마세요!
   - **중요**: 교통/접근성에 대한 구체적 언급이 없으면 반드시 `null`로 설정
   
   [키워드 추출 방법]
   1. **빈도수**가 높은 키워드 선택
   2. 서로 다른 의미의 키워드를 균형있게 포함
      예: "단풍"(긍정) + "낡은시설"(부정) 둘 다 포함 가능
   3. 극소수 의견(1~2명만 언급)은 제외

3. 감정 (sentiment_label)
   - **positive**: 전체의 70% 이상이 긍정적 표현 (추천, 재방문 의사, 만족, 행복)
   - **neutral**: 긍정/부정이 혼재되거나 중립적 표현 (단순 정보 전달, 감정 표현 절제)
   - **negative**: 전체의 70% 이상이 부정적 표현 (비추천, 실망, 불만, 위험 경고)
   ※ 20개 리뷰의 **전반적인 분위기**를 판단하세요.

4. 신뢰도 (reliability)
   - **high**: 직접 경험한 디테일이 풍부함, 구체적 언급이 10개 이상, 의견이 일관됨
   - **medium**: 정보는 있으나 평이하거나 일반적임, 의견이 다소 혼재
   - **low**: 내용이 모호하거나, 의견이 극단적으로 분산

[주의사항]
- **한국어 뉘앙스 파악**: "조금 아쉽다(부정 힌트)", "힘들지만 갈만하다(성취감 긍정)" 등 완곡한 표현과 이모티콘, 문맥을 깊이 있게 고려하세요.
- **키워드 추출**: 각 항목의 키워드는 해당 매력 포인트를 가장 잘 설명하는 명사/명사구로 구성하세요.
- **출력 형식**: 순수 JSON 문자열만 출력하세요.
- **JSON 문법 준수**: 모든 키(Key)와 문자열 값(Value)은 반드시 큰따옴표("")로 감싸야 합니다.

[출력 예시]
{
  "review_id": "가리산_naver_map_1",
  "overall_keywords": ["뷰맛집", "힐링코스", "초보추천", "단풍명소", "가을산행"],
  "view_score": 9,
  "view_keywords": ["정상뷰", "운해", "일출명소"],
  "healing_score": 8,
  "healing_keywords": ["숲길", "피톤치드"],
  "sns_photo_score": null,
  "sns_keywords": []],
  "trail_condition_score": 7,
  "trail_keywords": ["등산로정비", "이정표"],
  "fun_achievement_score": null,
  "fun_keywords": [],
  "seasonal_attraction_score": 10,
  "seasonal_keywords": ["단풍명소", "가을산", "억세"],
  "accessibility_score": 2,
  "accessibility_keywords": ["주차장없음", "도로혼잡", "주차힘듦"],
  "sentiment_label": "positive",
  "reliability": "high"
}


"""

# 클래스

In [ ]:
class HikingReviewFilterResult(BaseModel):
    review_id: str = Field(description="리뷰 ID")
    is_hiking_review: bool = Field(
        description="실질적인 등산/코스 후기이면 True, 아니면 False"
    )

# 감정 분류를 위한 Enum 정의
class SentimentType(str, enum.Enum):
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"

# 신뢰도 등급을 위한 Enum 정의
class ReliabilityLevel(str, enum.Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

# 단일 후기 분석 결과 스키마 (1~10 점수 + 항목별 키워드)
class AttractionReviewAnalysis(BaseModel):
    review_id: str = Field(description="리뷰 ID")
    
    # 👇 전체 핵심 키워드 추가
    overall_keywords: List[str] = Field(
        default_factory=list,
        description="전체 핵심 키워드 (최대 5개)",
        max_length=5
    )
    # 뷰·경관
    view_score: Optional[int] = Field(
        default=None, 
        description="뷰·경관 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    view_keywords: List[str] = Field(
        default_factory=list,
        description="뷰·경관 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 힐링 분위기
    healing_score: Optional[int] = Field(
        default=None, 
        description="힐링 분위기 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    healing_keywords: List[str] = Field(
        default_factory=list,
        description="힐링 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 사진/SNS 매력도
    sns_photo_score: Optional[int] = Field(
        default=None, 
        description="사진/SNS 매력도 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    sns_keywords: List[str] = Field(
        default_factory=list,
        description="사진/SNS 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 등산로 관리 상태
    trail_condition_score: Optional[int] = Field(
        default=None, 
        description="등산로 관리 상태 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    trail_keywords: List[str] = Field(
        default_factory=list,
        description="등산로 관리 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 재미·성취감
    fun_achievement_score: Optional[int] = Field(
        default=None, 
        description="재미·성취감 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    fun_keywords: List[str] = Field(
        default_factory=list,
        description="재미·성취감 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 계절별 매력도
    seasonal_attraction_score: Optional[int] = Field(
        default=None, 
        description="계절별 매력도 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    seasonal_keywords: List[str] = Field(
        default_factory=list,
        description="계절별 매력도 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 접근성 점수
    accessibility_score: Optional[int] = Field(
        default=None, 
        description="접근성 편의성 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    accessibility_keywords: List[str] = Field(
        default_factory=list,
        description="접근 편의성 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    sentiment_label: SentimentType = Field(description="감정 분류")
    reliability: ReliabilityLevel = Field(description="신뢰도 등급")

# 배치 분석을 위한 스키마
class BatchAttractionAnalysis(BaseModel):
    total_analyzed: int = Field(description="분석된 리뷰 총 수")
    analysis_results: List[AttractionReviewAnalysis] = Field(description="개별 리뷰 분석 결과") 

class YoutubeReviewFilterResult(BaseModel):
    review_id: str = Field(description="리뷰 ID")
    is_youtube_review: bool = Field(
        description="댓글이 산행/등산 경험이나 산 관련 정보(코스, 난이도, 풍경, 주변 시설 등)를 포함하면 True, 아니면 False"
    )
    
class YoutubeReviewAnalysis(BaseModel):
    review_id: str = Field(description="리뷰 ID")

    overall_keywords: List[str]
    
    # 뷰·경관
    view_score: Optional[int] = Field(
        default=None, 
        description="뷰·경관 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    view_keywords: List[str] = Field(
        default_factory=list,
        description="뷰·경관 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 힐링 분위기
    healing_score: Optional[int] = Field(
        default=None, 
        description="힐링 분위기 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    healing_keywords: List[str] = Field(
        default_factory=list,
        description="힐링 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 사진/SNS 매력도
    sns_photo_score: Optional[int] = Field(
        default=None, 
        description="사진/SNS 매력도 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    sns_keywords: List[str] = Field(
        default_factory=list,
        description="사진/SNS 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 등산로 관리 상태
    trail_condition_score: Optional[int] = Field(
        default=None, 
        description="등산로 관리 상태 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    trail_keywords: List[str] = Field(
        default_factory=list,
        description="등산로 관리 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 재미·성취감
    fun_achievement_score: Optional[int] = Field(
        default=None, 
        description="재미·성취감 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    fun_keywords: List[str] = Field(
        default_factory=list,
        description="재미·성취감 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    # 계절별 매력도
    seasonal_attraction_score: Optional[int] = Field(
        default=None, 
        description="계절별 매력도 점수 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    seasonal_keywords: List[str] = Field(
        default_factory=list,
        description="계절별 매력도 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    accessibility_score: Optional[int] = Field(
        default=None, 
        description="접근 편의성 (1~10), 언급 없으면 None", 
        ge=1, le=10
    )
    accessibility_keywords: List[str] = Field(
        default_factory=list,
        description="접근 편의성 핵심 키워드 (최대 3개)",
        max_length=3
    )
    
    sentiment_label: SentimentType = Field(description="감정 분류")
    reliability: ReliabilityLevel = Field(description="신뢰도 등급")

# 배치 분석을 위한 스키마
class BatchAttractionAnalysis_youtube(BaseModel):
    total_analyzed: int = Field(description="분석된 리뷰 총 수")
    analysis_results: List[YoutubeReviewAnalysis] = Field(description="개별 리뷰 분석 결과")

# 함수

In [ ]:
# 필터링 함수

def filter_hiking_review(review_text: str, review_id: Optional[str] = None) -> HikingReviewFilterResult:
    """
    등산 후기 여부를 필터링합니다.
    
    Args:
        review_text: 분석할 후기 텍스트
        review_id: 후기 고유 ID (선택사항)
    
    Returns:
        HikingReviewFilterResult: 필터링 결과 (review_id, is_hiking_review)
    """
    # 리뷰 ID가 없으면 타임스탬프 기반으로 생성
    if review_id is None:
        review_id = f"review_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # 프롬프트 구성
    prompt = f"""
리뷰 ID: {review_id}

후기 내용:
{review_text}

위 텍스트가 실질적인 등산 후기인지 판단하여 JSON 형식으로 출력하세요.
"""
    
    try:
        # Gemini API 호출 (필터링)
        response = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=filter_system_instruction,
                response_mime_type="application/json",
                response_schema=HikingReviewFilterResult,
                temperature=0.2
            )
        )
        # 응답 유효성 검사
        if not response:
            raise ValueError("API 응답 없음")
        
        if not hasattr(response, 'text'):
            raise ValueError("응답에 텍스트 없음")
        
        # 결과 파싱
        result = HikingReviewFilterResult.model_validate_json(response.text)
        return result
        
    
    except AttributeError as e:
        print(f"❌ 속성 오류: {e}")
        raise    
    except Exception as e:
        print(f"❌ 필터링 중 오류 발생: {e}")
        raise


# 상세 분석 함수

def analyze_single_review(review_text: str, review_id: Optional[str] = None) -> AttractionReviewAnalysis:
    """
    단일 등산 후기를 분석하여 구조화된 결과를 반환합니다.
    
    Args:
        review_text: 분석할 후기 텍스트
        review_id: 후기 고유 ID (선택사항)
    
    Returns:
        AttractionReviewAnalysis: 분석 결과 객체
    """
    # 리뷰 ID가 없으면 타임스탬프 기반으로 생성
    if review_id is None:
        review_id = f"review_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # 프롬프트 구성
    prompt = f"""
리뷰 ID: {review_id}

후기 내용:
{review_text}

위 등산 후기를 분석하여 JSON 형식으로 결과를 출력하세요.
"""
    
    try:
        # Gemini API 호출 (구조화된 출력)
        response = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=analysis_system_instruction,
                response_mime_type="application/json",
                response_schema=AttractionReviewAnalysis,
                temperature=0.2
            )
        )
        
        # 응답 유효성 검사
        if not response:
            raise ValueError("API 응답 없음")
        
        if not hasattr(response, 'text'):
            raise ValueError("응답에 텍스트 없음")
        
        # 결과 파싱
        result = AttractionReviewAnalysis.model_validate_json(response.text)
        return result
        
    except AttributeError as e:
        print(f"❌ 속성 오류: {e}")
        raise
    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        raise


# 통합 분석 함수 (필터링 + 상세 분석)

def analyze_single_review_with_filter(
        review_text: str, 
        review_id: Optional[str] = None
    ) -> Optional[AttractionReviewAnalysis]:
    """
    필터링 후 등산 후기인 경우에만 상세 분석을 수행합니다.
    
    Args:
        review_text: 분석할 후기 텍스트
        review_id: 후기 고유 ID (선택사항)
    
    Returns:
        AttractionReviewAnalysis 또는 None: 등산 후기가 아니면 None 반환
    """
    # 리뷰 ID가 없으면 타임스탬프 기반으로 생성
    if review_id is None:
        review_id = f"review_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # 1단계: 모델 B로 필터링
    filter_result = filter_hiking_review(review_text, review_id)
    
    # 2단계: 등산 후기인 경우에만 모델 A로 상세 분석
    if filter_result.is_hiking_review:
        print(f"    ✅ 등산 후기 확인 → 상세 분석 진행")
        return analyze_single_review(review_text, review_id)
    else:
        print(f"    ⏭️  등산 후기 아님 → 스킵")
        return None
    


# 배치 분석 함수 (재처리 기능 추가)

def analyze_batch_reviews(reviews: List[Dict[str, str]], use_filter: bool = True) -> BatchAttractionAnalysis:
    """
    여러 등산 후기를 일괄 분석하여 배치 결과를 반환합니다.
    
    Args:
        reviews: [{"review_id": "...", "text": "..."}, ...] 형태의 후기 리스트
        use_filter: True이면 모델 B로 필터링 후 분석, False이면 바로 분석
    
    Returns:
        BatchAttractionAnalysis: 배치 분석 결과
    """
    analysis_results = []
    filtered_count = 0
    failed_reviews = []  # 👈 실패한 리뷰 기록
    
    print(f"\n📊 총 {len(reviews)}개의 후기 분석 시작...")
    if use_filter:
        print("🔍 필터링 모드 활성화: 등산 후기만 상세 분석합니다.\n")
    else:
        print("⚡ 필터링 없이 전체 분석합니다.\n")
    
    # 청크 단위로 처리
    chunk_size = 10
    total_reviews = len(reviews)
    
    for chunk_start in range(0, total_reviews, chunk_size):
        chunk_end = min(chunk_start + chunk_size, total_reviews)
        chunk_reviews = reviews[chunk_start:chunk_end]
        
        print(f"\n[청크 {chunk_start+1}~{chunk_end}] 처리 중...")
        
        for idx, review in enumerate(chunk_reviews, start=chunk_start+1):
            review_id = review.get('review_id', f'review_{idx}')
            review_text = review.get('text', '')
            
            print(f"  [{idx}/{total_reviews}] 분석 중: {review_id}")
            
            try:
                if use_filter:
                    # 필터링 + 상세 분석
                    result = analyze_single_review_with_filter(review_text, review_id)
                    if result is None:
                        filtered_count += 1
                        continue
                    else:
                        analysis_results.append(result)
                else:
                    # 필터링 없이 바로 상세 분석
                    result = analyze_single_review(review_text, review_id)
                    analysis_results.append(result)
                    
            except AttributeError as e:
                print(f"    ⚠️ 속성 오류: {e}")
                # 👇 실패 정보 저장
                failed_reviews.append({
                    'review_id': review_id,
                    'text': review_text,
                    'error_type': 'AttributeError',
                    'error_message': str(e)
                })
                continue
            except Exception as e:
                print(f"    ⚠️ 분석 실패: {e}")
                # 👇 실패 정보 저장
                failed_reviews.append({
                    'review_id': review_id,
                    'text': review_text,
                    'error_type': type(e).__name__,
                    'error_message': str(e)
                })
                continue
        
        # 청크 완료 후 잠시 대기 (Rate Limit 방지)
        if chunk_end < total_reviews:
            print(f"  💤 다음 청크 전 대기 (1초)...")
            time.sleep(1)
    
    # 요약 출력
    print(f"\n✅ 배치 분석 완료!")
    print(f"   - 분석 완료: {len(analysis_results)}건")
    if use_filter and filtered_count > 0:
        print(f"   - 필터링: {filtered_count}건 (등산 후기 아님)")
    if failed_reviews:
        print(f"   - ⚠️ 실패: {len(failed_reviews)}건")
    
    batch_result = BatchAttractionAnalysis(
        total_analyzed=len(analysis_results),
        analysis_results=analysis_results
    )
    
    # 👇 실패한 리뷰 저장
    if failed_reviews:
        save_failed_reviews(failed_reviews)
    
    return batch_result


def save_failed_reviews(failed_reviews: List[Dict]):
    """실패한 리뷰를 JSON 파일로 저장"""
    
    filename = f"failed_reviews_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(failed_reviews, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 실패한 리뷰 저장: {filename}")
    print(f"   → 이 파일로 재처리 가능합니다!")

# 결과 출력 함수

def print_analysis_result(result: AttractionReviewAnalysis):
    """분석 결과를 보기 좋게 출력합니다."""
    print("\n" + "="*60)
    print(f"📝 리뷰 ID: {result.review_id}")

    # 👇 전체 핵심 키워드 추가
    overall_kw = ', '.join(result.overall_keywords) if result.overall_keywords else '-'
    print(f"🏷️  전체 핵심 키워드: {overall_kw}")
    
    print(f"😊 감정: {result.sentiment_label.value} | 신뢰도: {result.reliability.value}")
    print("-"*60)
    print("📊 매력 포인트 점수 (1~10) + 키워드:")
    print("-"*60)
    
    # 뷰·경관
    score_display = result.view_score if result.view_score else 'N/A'
    keywords_display = ', '.join(result.view_keywords) if result.view_keywords else '-'
    print(f"  🏔️  뷰·경관: {score_display} | {keywords_display}")
    
    # 힐링 분위기
    score_display = result.healing_score if result.healing_score else 'N/A'
    keywords_display = ', '.join(result.healing_keywords) if result.healing_keywords else '-'
    print(f"  🌲 힐링 분위기: {score_display} | {keywords_display}")
    
    # SNS 매력도
    score_display = result.sns_photo_score if result.sns_photo_score else 'N/A'
    keywords_display = ', '.join(result.sns_keywords) if result.sns_keywords else '-'
    print(f"  📸 SNS 매력도: {score_display} | {keywords_display}")
    
    # 등산로 관리
    score_display = result.trail_condition_score if result.trail_condition_score else 'N/A'
    keywords_display = ', '.join(result.trail_keywords) if result.trail_keywords else '-'
    print(f"  🛤️  등산로 관리: {score_display} | {keywords_display}")
    
    # 재미·성취감
    score_display = result.fun_achievement_score if result.fun_achievement_score else 'N/A'
    keywords_display = ', '.join(result.fun_keywords) if result.fun_keywords else '-'
    print(f"  🎯 재미·성취감: {score_display} | {keywords_display}")
    
    # 계절별 매력
    score_display = result.seasonal_attraction_score if result.seasonal_attraction_score else 'N/A'
    keywords_display = ', '.join(result.seasonal_keywords) if result.seasonal_keywords else '-'
    print(f"  🍂 계절별 매력: {score_display} | {keywords_display}")
    
    # 접근 편의성
    score_display = result.accessibility_score if result.accessibility_score else 'N/A'
    keywords_display = ', '.join(result.accessibility_keywords) if result.accessibility_keywords else '-'
    print(f"  🚘 접근 편의성: {score_display} | {keywords_display}")
    
    print("="*60)

def print_batch_summary(batch: BatchAttractionAnalysis):
    """배치 분석 결과를 요약하여 출력합니다."""
    print("\n" + "="*60)
    print(f"📦 배치 분석 요약")
    print("="*60)
    print(f"총 분석 건수: {batch.total_analyzed}건")
    
    if batch.total_analyzed > 0:
        # 평균 점수 계산
        def calc_avg(score_list):
            valid_scores = [s for s in score_list if s is not None]
            return sum(valid_scores) / len(valid_scores) if valid_scores else 0
        
        avg_view = calc_avg([r.view_score for r in batch.analysis_results])
        avg_healing = calc_avg([r.healing_score for r in batch.analysis_results])
        avg_sns = calc_avg([r.sns_photo_score for r in batch.analysis_results])
        avg_trail = calc_avg([r.trail_condition_score for r in batch.analysis_results])
        avg_fun = calc_avg([r.fun_achievement_score for r in batch.analysis_results])
        avg_seasonal = calc_avg([r.seasonal_attraction_score for r in batch.analysis_results])
        avg_acc = calc_avg([r.accessibility_score for r in batch.analysis_results])
        
        positive_count = sum(1 for r in batch.analysis_results if r.sentiment_label == SentimentType.POSITIVE)
        positive_ratio = positive_count / len(batch.analysis_results) * 100
        
        print(f"\n평균 점수 (1~10):")
        print(f"  뷰·경관: {avg_view:.1f}")
        print(f"  힐링: {avg_healing:.1f}")
        print(f"  SNS매력: {avg_sns:.1f}")
        print(f"  등산로관리: {avg_trail:.1f}")
        print(f"  재미·성취감: {avg_fun:.1f}")
        print(f"  계절매력: {avg_seasonal:.1f}")
        print(f"  접근 편의성: {avg_acc:.1f}")
        print(f"\n긍정 후기: {positive_ratio:.1f}%")
    
    print("="*60)



# CSV 데이터 로드 및 전처리 함수

def load_hiking_reviews_from_csv(csv_path: str) -> pd.DataFrame:
    """
    CSV 파일에서 등산 후기 데이터를 로드합니다.
    
    Args:
        csv_path: CSV 파일 경로
    
    Returns:
        DataFrame: 로드된 데이터프레임
    """
    try:
        df = pd.read_csv(csv_path)
        print(f"✅ CSV 파일 로드 완료: {len(df)}개 행")
        print(f"📊 컬럼: {list(df.columns)}")
        
        # full_content 컬럼 확인
        if 'full_content' not in df.columns:
            raise ValueError("CSV 파일에 'full_content' 컬럼이 없습니다!")
        
        # 결측치 제거
        df_cleaned = df.dropna(subset=['full_content'])
        print(f"🧹 결측치 제거 후: {len(df_cleaned)}개 행")
        
        return df_cleaned
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        raise
    except Exception as e:
        print(f"❌ CSV 로드 중 오류: {e}")
        raise

def analyze_csv_reviews(csv_path: str, max_reviews: Optional[int] = None, 
                        use_filter: bool = True, save_output: bool = True) -> BatchAttractionAnalysis:
    """
    CSV 파일의 full_content를 분석하여 결과를 반환합니다.
    
    Args:
        csv_path: CSV 파일 경로
        max_reviews: 분석할 최대 후기 수 (None이면 전체)
        use_filter: True이면 필터링 후 분석, False이면 전체 분석
        save_output: 결과를 CSV로 저장할지 여부
    
    Returns:
        BatchAttractionAnalysis: 배치 분석 결과
    """
    # CSV 로드
    df = load_hiking_reviews_from_csv(csv_path)
    
    # 분석할 데이터 선택
    if max_reviews:
        df = df.head(max_reviews)
        print(f"🎯 {max_reviews}개 후기만 분석합니다.")
    
    # 후기 리스트 생성
    reviews = []
    for idx, row in df.iterrows():
        review_id = str(row.get('review_id', f'review_{idx}'))  # id 컬럼이 있으면 사용
        review_text = str(row['full_content'])
        
        reviews.append({
            'review_id': review_id,
            'text': review_text
        })
    
    # 배치 분석 실행
    batch_result = analyze_batch_reviews(reviews, use_filter=use_filter)
    
    # 결과 저장
    if save_output:
        save_analysis_results(batch_result, csv_path)
    
    return batch_result

def save_analysis_results(batch_result: BatchAttractionAnalysis, original_csv_path: str):
    """
    분석 결과를 CSV 파일로 저장합니다.
    
    Args:
        batch_result: 배치 분석 결과
        original_csv_path: 원본 CSV 파일 경로 (저장 파일명 생성용)
    """
    # 결과를 DataFrame으로 변환
    results_data = []
    for result in batch_result.analysis_results:
        results_data.append({
            'review_id': result.review_id,
            'overall_keywords': ', '.join(result.overall_keywords) if result.overall_keywords else '',
            'view_score': result.view_score,
            'view_keywords': ', '.join(result.view_keywords) if result.view_keywords else '',
            'healing_score': result.healing_score,
            'healing_keywords': ', '.join(result.healing_keywords) if result.healing_keywords else '',
            'sns_photo_score': result.sns_photo_score,
            'sns_keywords': ', '.join(result.sns_keywords) if result.sns_keywords else '',
            'trail_condition_score': result.trail_condition_score,
            'trail_keywords': ', '.join(result.trail_keywords) if result.trail_keywords else '',
            'fun_achievement_score': result.fun_achievement_score,
            'fun_keywords': ', '.join(result.fun_keywords) if result.fun_keywords else '',
            'seasonal_attraction_score': result.seasonal_attraction_score,
            'seasonal_keywords': ', '.join(result.seasonal_keywords) if result.seasonal_keywords else '',
            'accessibility_score': result.accessibility_score,
            'accessibility_keywords': ', '.join(result.accessibility_keywords) if result.accessibility_keywords else '',
            'sentiment_label': result.sentiment_label.value,
            'reliability': result.reliability.value
        })
    
    results_df = pd.DataFrame(results_data)
    
    # 저장 파일명 생성
    base_name = os.path.splitext(original_csv_path)[0]
    output_path = f"{base_name}_분석결과_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    
    # CSV 저장
    results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n💾 분석 결과 저장 완료: {output_path}")
    
    # 요약 통계 저장 (None 값 제외하고 평균 계산)
    summary_path = f"{base_name}_요약_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("등산 후기 분석 요약 보고서\n")
        f.write("="*60 + "\n\n")
        f.write(f"총 분석 건수: {batch_result.total_analyzed}건\n\n")
        f.write("="*60 + "\n")
        f.write("평균 점수 (1~10 척도, N/A 제외)\n")
        f.write("="*60 + "\n")
        
        # None 값을 제외한 평균 계산
        def write_avg(name, column):
            valid_values = results_df[column].dropna()
            if len(valid_values) > 0:
                avg = valid_values.mean()
                f.write(f"{name}: {avg:.2f} (유효 데이터: {len(valid_values)}개)\n")
            else:
                f.write(f"{name}: N/A (데이터 없음)\n")
        
        write_avg("뷰·경관", 'view_score')
        write_avg("힐링 분위기", 'healing_score')
        write_avg("SNS 매력도", 'sns_photo_score')
        write_avg("등산로 관리", 'trail_condition_score')
        write_avg("재미·성취감", 'fun_achievement_score')
        write_avg("계절별 매력", 'seasonal_attraction_score')
        write_avg("접근 편의성", 'accessibility_score')
        
        f.write("\n" + "="*60 + "\n")
        f.write("감정 분포\n")
        f.write("="*60 + "\n")
        sentiment_counts = results_df['sentiment_label'].value_counts()
        for sentiment, count in sentiment_counts.items():
            percentage = count / len(results_df) * 100
            f.write(f"{sentiment}: {count}건 ({percentage:.1f}%)\n")
    
    print(f"💾 요약 보고서 저장 완료: {summary_path}")
    
    return results_df

def filter_youtube_review(review_text: str, review_id: Optional[str] = None) -> YoutubeReviewFilterResult:
    """
    등산 후기 여부를 필터링합니다.
    
    Args:
        review_text: 분석할 후기 텍스트
        review_id: 후기 고유 ID (선택사항)
    
    Returns:
        YoutubeReviewFilterResult: 필터링 결과 (review_id, is_hiking_review)
    """
    # 리뷰 ID가 없으면 타임스탬프 기반으로 생성
    if review_id is None:
        review_id = f"review_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # 프롬프트 구성
    prompt = f"""
리뷰 ID: {review_id}

후기 내용:
{review_text}

위 텍스트가 실질적인 등산 후기인지 판단하여 JSON 형식으로 출력하세요.
"""
    
    try:
        # Gemini API 호출 (필터링)
        response = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=filter_system_instruction_youtube,
                response_mime_type="application/json",
                response_schema=YoutubeReviewFilterResult,
                temperature=0.2
            )
        )
        # 응답 유효성 검사
        if not response:
            raise ValueError("API 응답 없음")
        
        if not hasattr(response, 'text'):
            raise ValueError("응답에 텍스트 없음")
        
        # 결과 파싱
        result = YoutubeReviewFilterResult.model_validate_json(response.text)
        return result
        
    
    except AttributeError as e:
        print(f"❌ 속성 오류: {e}")
        raise    
    except Exception as e:
        print(f"❌ 필터링 중 오류 발생: {e}")
        raise

def analyze_single_review_youtube(review_text: str, review_id: Optional[str] = None) -> YoutubeReviewAnalysis:
    """
    단일 등산 후기를 분석하여 구조화된 결과를 반환합니다.
    
    Args:
        review_text: 분석할 후기 텍스트
        review_id: 후기 고유 ID (선택사항)
    
    Returns:
        YoutubeReviewAnalysis: 분석 결과 객체
    """
    # 리뷰 ID가 없으면 타임스탬프 기반으로 생성
    if review_id is None:
        review_id = f"review_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # 프롬프트 구성
    prompt = f"""
리뷰 ID: {review_id}

후기 내용:
{review_text}

위 등산 후기를 분석하여 JSON 형식으로 결과를 출력하세요.
"""
    
    try:
        # Gemini API 호출 (구조화된 출력)
        response = client.models.generate_content(
            model=gemini_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=analysis_system_instruction_youtube,
                response_mime_type="application/json",
                response_schema=YoutubeReviewAnalysis,
                temperature=0.2
            )
        )
        
        # 응답 유효성 검사
        if not response:
            raise ValueError("API 응답 없음")
        
        if not hasattr(response, 'text'):
            raise ValueError("응답에 텍스트 없음")
        
        # 결과 파싱
        result = YoutubeReviewAnalysis.model_validate_json(response.text)
        return result
        
    except AttributeError as e:
        print(f"❌ 속성 오류: {e}")
        raise
    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        raise

def analyze_single_review_with_filter_youtube(
        review_text: str, 
        review_id: Optional[str] = None
        ) -> Optional[YoutubeReviewAnalysis]:
    """
    필터링 후 산 관련 내용인 경우에만 상세 분석을 수행합니다.
    
    Args:
        review_text: 분석할 후기 텍스트
        review_id: 후기 고유 ID (선택사항)
    
    Returns:
        YoutubeReviewAnalysis 또는 None: 산관련 내용이 아니면 None 반환
    """
    # 리뷰 ID가 없으면 타임스탬프 기반으로 생성
    if review_id is None:
        review_id = f"review_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # 1단계: 모델 B로 필터링
    filter_result = filter_youtube_review(review_text, review_id)
    
    # 2단계: 등산 후기인 경우에만 모델 A로 상세 분석
    if filter_result.is_youtube_review:
        print(f"    ✅ 산 관련 내용 확인 → 상세 분석 진행")
        return analyze_single_review_youtube(review_text, review_id)
    else:
        print(f"    ⏭️  산 관련 내용 아님 → 스킵")
        return None

def analyze_batch_reviews_youtube(reviews: List[Dict[str, str]], use_filter: bool = True) -> BatchAttractionAnalysis_youtube:
    """
    여러 등산 후기를 일괄 분석하여 배치 결과를 반환합니다.
    
    Args:
        reviews: [{"review_id": "...", "text": "..."}, ...] 형태의 후기 리스트
        use_filter: True이면 모델 B로 필터링 후 분석, False이면 바로 분석
    
    Returns:
        BatchAttractionAnalysis_youtube: 배치 분석 결과
    """
    analysis_results = []
    filtered_count = 0
    failed_reviews = []  # 👈 실패한 리뷰 기록
    
    print(f"\n📊 총 {len(reviews)}개의 후기 분석 시작...")
    if use_filter:
        print("🔍 필터링 모드 활성화: 등산 후기만 상세 분석합니다.\n")
    else:
        print("⚡ 필터링 없이 전체 분석합니다.\n")
    
    # 청크 단위로 처리
    chunk_size = 10
    total_reviews = len(reviews)
    
    for chunk_start in range(0, total_reviews, chunk_size):
        chunk_end = min(chunk_start + chunk_size, total_reviews)
        chunk_reviews = reviews[chunk_start:chunk_end]
        
        print(f"\n[청크 {chunk_start+1}~{chunk_end}] 처리 중...")
        
        for idx, review in enumerate(chunk_reviews, start=chunk_start+1):
            review_id = review.get('review_id', f'review_{idx}')
            review_text = review.get('text', '')
            
            print(f"  [{idx}/{total_reviews}] 분석 중: {review_id}")
            
            try:
                if use_filter:
                    # 필터링 + 상세 분석
                    result = analyze_single_review_with_filter_youtube(review_text, review_id)
                    if result is None:
                        filtered_count += 1
                        continue
                    else:
                        analysis_results.append(result)
                else:
                    # 필터링 없이 바로 상세 분석
                    result = analyze_single_review_youtube(review_text, review_id)
                    analysis_results.append(result)
                    
            except Exception as e:
                print(f"    ⚠️ 분석 실패: {e}")
                # 👇 실패 정보 저장
                failed_reviews.append({
                    'review_id': review_id,
                    'text': review_text,
                    'error_type': type(e).__name__,
                    'error_message': str(e)
                })
                continue
        
        # 청크 완료 후 잠시 대기 (Rate Limit 방지)
        if chunk_end < total_reviews:
            print(f"  💤 다음 청크 전 대기 (1초)...")
            time.sleep(1)
    
    # 요약 출력
    print(f"\n✅ 배치 분석 완료!")
    print(f"   - 분석 완료: {len(analysis_results)}건")
    if use_filter and filtered_count > 0:
        print(f"   - 필터링: {filtered_count}건 (등산 후기 아님)")
    if failed_reviews:
        print(f"   - ⚠️ 실패: {len(failed_reviews)}건")
    
    batch_result = BatchAttractionAnalysis_youtube(
        total_analyzed=len(analysis_results),
        analysis_results=analysis_results
    )
    
    # 👇 실패한 리뷰 저장
    if failed_reviews:
        save_failed_reviews(failed_reviews)
    
    return batch_result

def print_analysis_result_youtube(result: YoutubeReviewAnalysis):
    """분석 결과를 보기 좋게 출력합니다."""
    print("\n" + "="*60)
    print(f"📝 리뷰 ID: {result.review_id}")

    # 👇 전체 핵심 키워드 추가
    overall_kw = ', '.join(getattr(result, 'overall_keywords', [])) or None
    print(f"🏷️  전체 핵심 키워드: {overall_kw}")


    def enum_to_str(v):
        return v.value if hasattr(v, "value") else v

    print(f"😊 감정: {enum_to_str(result.sentiment_label)} | "f"신뢰도: {enum_to_str(result.reliability)}")

    print("-"*60)
    print("📊 매력 포인트 점수 (1~10) + 키워드:")
    print("-"*60)
    


    # 뷰·경관
    score_display = getattr(result, 'view_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'view_keywords', [])) or None
    print(f"  🏔️  뷰·경관: {score_display} | {keywords_display}")

    # 힐링 분위기
    score_display = getattr(result, 'healing_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'healing_keywords', [])) or None
    print(f"  🌲 힐링 분위기: {score_display} | {keywords_display}")

    # SNS 매력도
    score_display = getattr(result, 'sns_photo_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'sns_keywords', [])) or None
    print(f"  📸 SNS 매력도: {score_display} | {keywords_display}")

    # 등산로 관리
    score_display = getattr(result, 'trail_condition_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'trail_keywords', [])) or None
    print(f"  🛤️  등산로 관리: {score_display} | {keywords_display}")

    # 재미·성취감
    score_display = getattr(result, 'fun_achievement_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'fun_keywords', [])) or None
    print(f"  🎯 재미·성취감: {score_display} | {keywords_display}")

    # 계절별 매력
    score_display = getattr(result, 'seasonal_attraction_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'seasonal_keywords', [])) or None
    print(f"  🍂 계절별 매력: {score_display} | {keywords_display}")

    # 접근 편의성
    score_display = getattr(result, 'accessibility_score', 'N/A')
    keywords_display = ', '.join(getattr(result, 'accessibility_keywords', [])) or None
    print(f"  🚘 접근 편의성: {score_display} | {keywords_display}")

    
    print("="*60)

def print_batch_summary_youtube(batch: BatchAttractionAnalysis_youtube):
    """배치 분석 결과를 요약하여 출력합니다."""
    print("\n" + "="*60)
    print(f"📦 배치 분석 요약")
    print("="*60)
    print(f"총 분석 건수: {batch.total_analyzed}건")
    
    if batch.total_analyzed > 0:
        # 평균 점수 계산
        def calc_avg(score_list):
            valid_scores = [s for s in score_list if s is not None]
            return sum(valid_scores) / len(valid_scores) if valid_scores else 0
        
        avg_view = calc_avg([r.view_score for r in batch.analysis_results])
        avg_healing = calc_avg([r.healing_score for r in batch.analysis_results])
        avg_sns = calc_avg([r.sns_photo_score for r in batch.analysis_results])
        avg_trail = calc_avg([r.trail_condition_score for r in batch.analysis_results])
        avg_fun = calc_avg([r.fun_achievement_score for r in batch.analysis_results])
        avg_seasonal = calc_avg([r.seasonal_attraction_score for r in batch.analysis_results])
        avg_acc = calc_avg([r.accessibility_score for r in batch.analysis_results])
        
        def is_positive(v):
            return v == SentimentType.POSITIVE or v == "positive"

        positive_count = sum(
            1 for r in batch.analysis_results
            if is_positive(r.sentiment_label))

        positive_ratio = positive_count / len(batch.analysis_results) * 100
        
        print(f"\n평균 점수 (1~10):")
        print(f"  뷰·경관: {avg_view:.1f}")
        print(f"  힐링: {avg_healing:.1f}")
        print(f"  SNS매력: {avg_sns:.1f}")
        print(f"  등산로관리: {avg_trail:.1f}")
        print(f"  재미·성취감: {avg_fun:.1f}")
        print(f"  계절매력: {avg_seasonal:.1f}")
        print(f"  접근 편의성: {avg_acc:.1f}")
        print(f"\n긍정 후기: {positive_ratio:.1f}%")
    
    print("="*60)

def load_youtube_reviews_from_csv(csv_path: str) -> pd.DataFrame:
    """
    CSV 파일에서 등산 후기 데이터를 로드합니다.
    
    Args:
        csv_path: CSV 파일 경로
    
    Returns:
        DataFrame: 로드된 데이터프레임
    """
    try:
        df = pd.read_csv(csv_path)
        print(f"✅ CSV 파일 로드 완료: {len(df)}개 행")
        print(f"📊 컬럼: {list(df.columns)}")
        
        # combined_full_content 컬럼 확인
        if 'combined_full_content' not in df.columns:
            raise ValueError("CSV 파일에 'combined_full_content' 컬럼이 없습니다!")
        
        # 결측치 제거
        df_cleaned = df.dropna(subset=['combined_full_content'])
        print(f"🧹 결측치 제거 후: {len(df_cleaned)}개 행")
        
        return df_cleaned
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        raise
    except Exception as e:
        print(f"❌ CSV 로드 중 오류: {e}")
        raise

def analyze_csv_reviews_youtube(csv_path: str, max_reviews: Optional[int] = None, 
                        use_filter: bool = True, save_output: bool = True) -> BatchAttractionAnalysis_youtube:
    """
    CSV 파일의 combined_full_content 분석하여 결과를 반환합니다.
    
    Args:
        csv_path: CSV 파일 경로
        max_reviews: 분석할 최대 후기 수 (None이면 전체)
        use_filter: True이면 필터링 후 분석, False이면 전체 분석
        save_output: 결과를 CSV로 저장할지 여부
    
    Returns:
        BatchAttractionAnalysis_youtube: 배치 분석 결과
    """
    # CSV 로드
    df = load_youtube_reviews_from_csv(csv_path)
    
    # 분석할 데이터 선택
    if max_reviews:
        df = df.head(max_reviews)
        print(f"🎯 {max_reviews}개 후기만 분석합니다.")
    
    # 후기 리스트 생성
    reviews = []
    for idx, row in df.iterrows():
        review_id = str(row.get('review_id', f'review_{idx}'))  # id 컬럼이 있으면 사용
        review_text = str(row['combined_full_content'])
        
        reviews.append({
            'review_id': review_id,
            'text': review_text
        })
    
    # 배치 분석 실행
    batch_result = analyze_batch_reviews_youtube(reviews, use_filter=use_filter)
    
    # 결과 저장
    if save_output:
        save_analysis_results(batch_result, csv_path)
    
    return batch_result

def save_analysis_results(batch_result: BatchAttractionAnalysis_youtube, original_csv_path: str):
    """
    분석 결과를 CSV 파일로 저장합니다.
    
    Args:
        batch_result: 배치 분석 결과
        original_csv_path: 원본 CSV 파일 경로 (저장 파일명 생성용)
    """
    # 결과를 DataFrame으로 변환
    results_data = []
    for result in batch_result.analysis_results:
        results_data.append({
        'review_id': getattr(result, 'review_id', None),
        'overall_keywords': ', '.join(getattr(result, 'overall_keywords', [])) or None,
        'view_score': getattr(result, 'view_score', None),
        'view_keywords': ', '.join(getattr(result, 'view_keywords', [])) or None,
        'healing_score': getattr(result, 'healing_score', None),
        'healing_keywords': ', '.join(getattr(result, 'healing_keywords', [])) or None,
        'sns_photo_score': getattr(result, 'sns_photo_score', None),
        'sns_keywords': ', '.join(getattr(result, 'sns_keywords', [])) or None,
        'trail_condition_score': getattr(result, 'trail_condition_score', None),
        'trail_keywords': ', '.join(getattr(result, 'trail_keywords', [])) or None,
        'fun_achievement_score': getattr(result, 'fun_achievement_score', None),
        'fun_keywords': ', '.join(getattr(result, 'fun_keywords', [])) or None,
        'seasonal_attraction_score': getattr(result, 'seasonal_attraction_score', None),
        'seasonal_keywords': ', '.join(getattr(result, 'seasonal_keywords', [])) or None,
        'accessibility_score': getattr(result, 'accessibility_score', None),
        'accessibility_keywords': ', '.join(getattr(result, 'accessibility_keywords', [])) or None,
        'sentiment_label': getattr(result, 'sentiment_label', None).value if getattr(result, 'sentiment_label', None) else None,
        'reliability': getattr(result, 'reliability', None).value if getattr(result, 'reliability', None) else None
    })
        
    results_df = pd.DataFrame(results_data)
    
    # 저장 파일명 생성
    base_name = os.path.splitext(original_csv_path)[0]
    output_path = f"{base_name}_분석결과_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    
    # CSV 저장
    results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n💾 분석 결과 저장 완료: {output_path}")
    
    # 요약 통계 저장 (None 값 제외하고 평균 계산)
    summary_path = f"{base_name}_요약_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("등산 후기 분석 요약 보고서\n")
        f.write("="*60 + "\n\n")
        f.write(f"총 분석 건수: {batch_result.total_analyzed}건\n\n")
        f.write("="*60 + "\n")
        f.write("평균 점수 (1~10 척도, N/A 제외)\n")
        f.write("="*60 + "\n")
        
        # None 값을 제외한 평균 계산
        def write_avg(name, column):
            valid_values = results_df[column].dropna()
            if len(valid_values) > 0:
                avg = valid_values.mean()
                f.write(f"{name}: {avg:.2f} (유효 데이터: {len(valid_values)}개)\n")
            else:
                f.write(f"{name}: N/A (데이터 없음)\n")
        
        write_avg("뷰·경관", 'view_score')
        write_avg("힐링 분위기", 'healing_score')
        write_avg("SNS 매력도", 'sns_photo_score')
        write_avg("등산로 관리", 'trail_condition_score')
        write_avg("재미·성취감", 'fun_achievement_score')
        write_avg("계절별 매력", 'seasonal_attraction_score')
        write_avg("접근 편의성", 'accessibility_score')
        
        f.write("\n" + "="*60 + "\n")
        f.write("감정 분포\n")
        f.write("="*60 + "\n")
        sentiment_counts = results_df['sentiment_label'].value_counts()
        for sentiment, count in sentiment_counts.items():
            percentage = count / len(results_df) * 100
            f.write(f"{sentiment}: {count}건 ({percentage:.1f}%)\n")
    
    print(f"💾 요약 보고서 저장 완료: {summary_path}")
    
    return results_df

## 1. 네이버 블로그 NLP

---

In [ ]:
# ==================================================
# 네이버 블로그만 필터링
# ==================================================

df = pd.read_csv("99대명산_크롤링.csv")
df_pre = df[df['data_source'] == "naver_blog"]
df_pre.to_csv("99대명산_크롤링_네이버블로그.csv", index= False)

In [ ]:
# ==================================================
# 사용 예시
# ==================================================
if __name__ == "__main__":
    # CSV 파일 경로 설정
    csv_file_path = "99대명산_크롤링_네이버블로그.csv"
    
    print("\n" + "="*60)
    print("🏔️  등산 후기 분석 시스템 시작 (1~10 점수 체계)")
    print("="*60)
    
    # CSV 파일 존재 확인
    if not os.path.exists(csv_file_path):
        print(f"\n❌ 파일을 찾을 수 없습니다: {csv_file_path}")
        print("현재 디렉토리에 '99대명산크롤링_수정.csv' 파일이 있는지 확인해주세요.")
    else:
        # 옵션 1: 전체 분석 (시간이 오래 걸릴 수 있음)
        batch_result = analyze_csv_reviews(csv_file_path, max_reviews=None, save_output=True)
        
        # # 옵션 2: 일부만 테스트 (처음 10개만)
        # print("\n🧪 테스트 모드: 처음 10개 후기만 분석합니다.")
        # print("전체 분석을 원하시면 코드에서 max_reviews=None으로 변경하세요.\n")
        
        # batch_result = analyze_csv_reviews(csv_file_path, max_reviews=10, save_output=True)
        
        # 결과 출력
        print_batch_summary(batch_result)
        
        print("\n\n📋 처음 3개 분석 결과 미리보기:")
        for i, result in enumerate(batch_result.analysis_results[:3], 1):
            print(f"\n--- {i}번째 후기 ---")
            print_analysis_result(result)


## 2. 네이버지도 / 카카오맵 리뷰 NLP

---


In [ ]:
# ==================================================
# 네이버 맵/ 카카오 맵 리뷰만 필터링
# ==================================================

df = pd.read_csv("99대명산_크롤링.csv")

df_pre = df[df["data_source"].isin(["naver_map", "kakao_map"])]
df_pre.to_csv("99대명산_지도.csv", index=False)

In [ ]:
# =======================================================
# 리뷰 텍스트가 짧아 20개씩 묶어 진행
# =======================================================

csv_file = "99대명산_지도.csv"
df = pd.read_csv(csv_file)

grouped_texts = []

for mountain, group in df.groupby("mountain_name"):
    num_splits = math.ceil(len(group) / 20)

    for i in range(num_splits):
        batch = group.iloc[i*20 : (i+1)*20]

        combined_text = " ".join(
            batch["full_content"]
            .dropna()
            .astype(str)
        )

        grouped_texts.append({
            "review_id": f"{mountain}_{i+1}",
            "mountain_name": mountain,
            "batch_index": i + 1,
            "collected_comment_count": len(batch),  # ⭐ 핵심
            "combined_full_content": combined_text
        })

result_df = pd.DataFrame(grouped_texts)
result_df.to_csv("99대명산_크롤링_지도.csv", index=False, encoding="utf-8-sig")

print("✅ 배치별 수집 댓글 수 포함 CSV 저장 완료")

In [ ]:
# ==================================================
# 사용 예시
# ==================================================
if __name__ == "__main__":
    # CSV 파일 경로 설정
    csv_file_path = "../data/99대명산_크롤링_지도.csv"
    
    print("\n" + "="*60)
    print("🏔️  등산 후기 분석 시스템 시작 (1~10 점수 체계)")
    print("="*60)
    
    # CSV 파일 존재 확인
    if not os.path.exists(csv_file_path):
        print(f"\n❌ 파일을 찾을 수 없습니다: {csv_file_path}")
        print("현재 디렉토리에 '../data/99대명산_크롤링_지도.csv' 파일이 있는지 확인해주세요.")
    else:
        # 옵션 1: 전체 분석 (시간이 오래 걸릴 수 있음)
        batch_result = analyze_csv_reviews(csv_file_path, max_reviews=None, save_output=True)
        
        # 옵션 2: 일부만 테스트 (처음 10개만)
        # print("\n🧪 테스트 모드: 처음 10개 후기만 분석합니다.")
        # print("전체 분석을 원하시면 코드에서 max_reviews=None으로 변경하세요.\n")
        
        # batch_result = analyze_csv_reviews(csv_file_path, max_reviews=10, save_output=True)
        
        # 결과 출력
        print_batch_summary(batch_result)
        
        print("\n\n📋 처음 3개 분석 결과 미리보기:")
        for i, result in enumerate(batch_result.analysis_results[:3], 1):
            print(f"\n--- {i}번째 후기 ---")
            print_analysis_result(result)

# 3. 유튜브 댓글 NLP

---

In [ ]:
# =======================================================
# 3. 유튜브 댓글 NLP
# =======================================================

# 1️⃣ CSV 불러오기
df = pd.read_csv("99대명산_크롤링.csv", encoding="utf-8-sig")  # 인코딩은 CSV에 맞게 조정

# 2️⃣ data_source가 "youtube"인 행만 필터링
df_youtube = df[df['data_source'] == 'youtube']

# 3️⃣ 새 CSV로 저장
df_youtube.to_csv("99대명산_youtube.csv", index=False, encoding="utf-8-sig")

print(f"총 {len(df_youtube)}개의 유튜브 데이터가 저장되었습니다.")

In [ ]:
# =======================================================
# 리뷰 텍스트가 짧아 20개씩 묶어 진행
# =======================================================

csv_file = "99대명산_youtube.csv"
df = pd.read_csv(csv_file)

grouped_texts = []

for mountain, group in df.groupby("mountain_name"):
    num_splits = math.ceil(len(group) / 20)

    for i in range(num_splits):
        batch = group.iloc[i*20 : (i+1)*20]

        combined_text = " ".join(
            batch["full_content"]
            .dropna()
            .astype(str)
        )

        grouped_texts.append({
            "review_id": f"{mountain}_{i+1}",
            "mountain_name": mountain,
            "batch_index": i + 1,
            "collected_comment_count": len(batch),  # ⭐ 핵심
            "combined_full_content": combined_text
        })

result_df = pd.DataFrame(grouped_texts)
result_df.to_csv("99대명산_크롤링_youtube.csv", index=False, encoding="utf-8-sig")

print("✅ 배치별 수집 댓글 수 포함 CSV 저장 완료")

In [ ]:
# ==================================================
# 메인 실행부
# ==================================================
if __name__ == "__main__":

    # CSV 파일 경로
    csv_file_path = "99대명산_크롤링_youtube.csv"

    print("\n" + "=" * 60)
    print("🏔️  등산 후기 분석 시스템 시작 (1~10 점수 체계)")
    print("=" * 60)

    # CSV 파일 존재 확인
    if not os.path.exists(csv_file_path):
        print(f"\n❌ 파일을 찾을 수 없습니다: {csv_file_path}")
        print("현재 디렉토리에 '99대명산_크롤링_youtube.csv' 파일이 있는지 확인해주세요.")
        exit(1)

    print("\n📂 분석 대상 파일:", csv_file_path)
    print("🚀 전체 후기 배치 분석을 시작합니다...\n")

    # ==================================================
    # 전체 분석 실행 (모든 배치)
    # ==================================================
    batch_result = analyze_csv_reviews_youtube(
        csv_file_path,
        max_reviews=None,      # ⭐ 전체 분석
        save_output=True       # 결과 CSV 저장
    )

    # ==================================================
    # 분석 요약 출력
    # ==================================================
    print_batch_summary_youtube(batch_result)

    # ==================================================
    # 샘플 결과 미리보기
    # ==================================================
    print("\n\n📋 분석 결과 샘플 미리보기 (처음 3개 배치):")

    for i, result in enumerate(batch_result.analysis_results[:3], 1):
        print(f"\n--- {i}번째 배치 ---")
        print(f"산 이름        : {result.mountain_name}")
        print(f"배치 ID        : {result.review_id}")
        print(f"수집 댓글 수   : {result.collected_comment_count}")
        print_analysis_result_youtube(result)

    print("\n✅ 전체 분석 완료!")


In [ ]:
 # 관광인프라 점수 산출로 넘어감